© 2026 Mechanome Bio Ltd. Proprietary & Confidential

# MB_05.2 — Spatial regression, **v2 extended** (mechanics + cell shape)

**Note: this notebook uses R (`mechmap-stats` kernel).**

Built on the MechanoMap **V5** `MB_05_spatial_regression.ipynb` pipeline, extended to test
**cell shape / morphology** metrics alongside the four production mechanical metrics.

This notebook **supersedes** `MB_05.2_spatial_regression_extended.ipynb` (March 2025, run on
pancreas tile R3C14), whose gSEM implementation was found to be incorrect during the
April 2026 gSEM QC review (`gSEM QC/MB_05_gSEM_technical_report.pdf`). Do not use the old
notebook or the results under `analysis_extended/R3C14/gsem_output/`.

## What was wrong with the old MB_05.2, and what v2 does instead

| # | Old MB_05.2 | v2 extended |
|---|---|---|
| 1 | **Linear XY detrending** — `lm(y ~ 1 + x + y)` removes only a flat first-order gradient, leaving glands, stromal bands and cell-type patches as uncontrolled spatial confounders | Thin-plate regression spline `s(centroid_x, centroid_y, k = K_SPATIAL)` via `mgcv::bam()` — the V5 gSEM smooth |
| 2 | **Cell ID matching bug** — the string `V1` column (`"cell_1"`, `"cell_2"`…) was used as the VMSI identifier when the integer `label` column is what `_relabeled_edges.csv` keys on, giving zero edge matches and junk tension statistics | Tension metrics are dropped (they had ρ≈0 and ~0 hits in the three-way QC); VMSI↔Xenium matching is the V5 k-d tree nearest-neighbour join on centroids |
| 3 | **No cell-area covariate** — cell size confounds both expression and geometry | `INCLUDE_AREA_COV` adds `s(log_cell_area, k = K_AREA_COV)` to every smooth |
| 4 | **Library-size normalisation** of counts | Xenium `cell_area` normalisation from `cells.csv.gz` (V5 default), library-size as fallback |
| 5 | **No transcript QC** (`MIN_EXPR_FRAC = 0`, no QV filter) | `QV_MIN`, negative-control/blank-probe filtering, `EXPR_FRAC_MIN` |
| 6 | **No VMSI validity filter** — mechanically unsolved cells (`pressure = 0`) entered the regression | Degenerate cells dropped before fitting, jointly from mechanics and GEX |
| 7 | **No winsorisation**, single global FDR across all metrics | `WINSORIZE_FRAC` on heavy-tailed metrics, `PADJ_PER_METRIC` BH correction per metric |
| 8 | **Single hard-coded tile**, absolute Windows path | Multi-tile stitching with overlap de-duplication; blank-template config |
| 9 | **`log()` applied blindly to every metric** — fine for stress, but `orient_alignment` is exactly 0 whenever the stress and shape axes are orthogonal, so `log(0) = -Inf` kills the metric | Per-metric `METRIC_TRANSFORM` table (`log` / `log1p` / `identity`), with a parity check asserting the mechanical path is still bit-for-bit the V5 result |

## Metrics tested

**Mechanical (unchanged from V5 production — results remain comparable to the 13 MechanoMap runs):**

| Metric | Source | Description |
|---|---|---|
| `pressure` | VMSI | Cell hydrostatic pressure (area-constraint Lagrange multiplier) |
| `isotropic_stress` | derived | `(λ₁ + λ₂) / 2` of the Cauchy stress tensor |
| `anisotropic_stress` | VMSI | Deviatoric stress magnitude |
| `anisotropy_fraction` | derived | `(λ₁ − λ₂) / (λ₁ + λ₂)` — dimensionless directional loading |

**Cell shape / morphology (new in v2), from VMSI vertex-model geometry:**

| Metric | Definition | Description |
|---|---|---|
| `shape_index` | `p / √A` | Vertex-model fluidity parameter; rigidity transition at ≈ 3.81 |
| `elongation` | `I₁ / I₂` | Inertia-tensor aspect ratio |
| `circularity` | `4πA / p²` | Cell roundness (1 = perfect circle) |
| `orient_alignment` | `\|cos(2(θ_stress − θ_shape))\|` | Stress-vs-long-axis co-alignment (1 = aligned, 0 = orthogonal) |

`shape_index` and `circularity` are scale-invariant by construction, so they are not collinear
with the `log_cell_area` covariate — the collinearity cell below verifies this on real data.

No extra input files are needed: all four come from the same
`{tile}_relabeled_cell_results.csv` that MB_02 already writes.

## Precedent

The shape extension was trialled once, on pancreas tile R3C14 (`gSEM QC`, April 2026).
Across the three method variants, `shape_index` returned 9 / 16 / 15 significant genes —
comparable to `pressure` at 12 / 13 / 25 — with cross-method Spearman ρ = 0.86, i.e. as
stable as the stress metrics. `circularity` gave 0 / 15 / 15, `elongation` 4 / 5 / 9.
`mean_tension` and `cv_tension` gave ρ ≈ 0 and are **not** carried into v2.

### Configuration

Edit only the **Required** block below to point at a new slide / tile set. Everything else is derived from `XENIUM_DIR` and `TILE_NAMES`.

- `XENIUM_DIR` &mdash; root of the Xenium output that already has MB_02 (`run_tensionmap`) results under `analysis/{TileID}/vmsi_output/` and per-tile transcript files under `tissue_tiles_transcripts/`.
- `TILE_NAMES` &mdash; character vector of tile basenames (e.g. `tile_R10_C14_Y19000_X26600`). Use one tile for a trial run; supply many for a real slide-wide run. The notebook stitches them, then runs gSEM **once** on the combined data so multiple-testing correction stays valid at the sample level.
> **v2 extended:** the shape-metric block at the bottom of the config cell is the only
> addition. Leave `INCLUDE_SHAPE_METRICS <- FALSE` to reproduce a stock V5 production run.

In [ ]:
# ===== Required =====
XENIUM_DIR   <- "/path/to/xenium_outs"
ANALYSIS_DIR <- file.path(XENIUM_DIR, "MB_ANALYSIS")
TILE_NAMES   <- c("tile_R0_C0_Y0_X0")  # full slide — single tile
TILE_PX  <- 8000   # pixels per tile; must match MB_01 TILE_SIZE
OVERLAP  <- 100   # must match MB_01 OVERLAP

# ===== Derived layout (matches MB_02 conventions) =====
tile_paths <- function(tile, root = ANALYSIS_DIR) {
  tile_id <- sub("^tile_(R\\d+)_(C\\d+)_(.+)$", "\\1\\2_\\3", tile)
  y_str   <- sub("^tile_R\\d+_C\\d+_Y(\\d+)_X\\d+$", "\\1", tile)
  x_str   <- sub("^tile_R\\d+_C\\d+_Y\\d+_X(\\d+)$", "\\1", tile)
  list(
    tile        = tile,
    tile_id     = tile_id,
    y_off       = as.integer(y_str),
    x_off       = as.integer(x_str),
    tensionmap  = file.path(root, "analysis", tile_id, "vmsi_output",
                            paste0(tile, "_relabeled_cell_results.csv")),
    transcripts = file.path(root, "tissue_tiles_transcripts",
                            paste0(tile, "_transcripts.csv.gz"))
  )
}

SAMPLE_NAME <- basename(ANALYSIS_DIR)
OUTPUT_DIR  <- file.path(ANALYSIS_DIR, "spatial_regression_1")

# ===== Normalisation =====
# cells.csv.gz lives in the Xenium outs root (XENIUM_DIR)
NORM_METHOD   <- "area"   # "area" uses Xenium cell_area; "library_size" uses total transcripts
MIN_CELL_AREA <- NULL     # NULL = 1st-percentile auto floor; numeric = hard floor in µm²

# ===== Regression params (DEFAULT = Variant E) =====
EXPR_FRAC_MIN  <- 0.05      # gene must be detected in >= this fraction of cells
N_GENES_SAMPLE <- NULL      # NULL = all expressed genes; integer = random subset for trials
RANDOM_SEED    <- 42
K_SPATIAL      <- 300        # Variant A: 30
MODEL_FX       <- FALSE      # Variant A: TRUE
USE_BAM        <- TRUE      # TRUE = bam(discrete=TRUE) for speed; FALSE = exact gam()
gSEM_METHOD    <- "fREML"   # "REML" (bam) or "fREML" (bam with discrete=TRUE)
NCORES         <- 6L         # fork-based parallel workers; adjust to your machine; each needs ~400-600 MB bam() transient
                             # (safe on 64 GB: 16 × 600 MB private + ~7 GB shared ≈ 17 GB peak)
CHECKPOINT_EVERY <- 100L    # write results to disk every N genes (enables crash recovery)
CHECKPOINT_FILE  <- file.path(OUTPUT_DIR, paste0(SAMPLE_NAME, "_gsem_checkpoint.csv"))
RESIDUALS_DIR    <- file.path(OUTPUT_DIR, "gsem_residuals")  # set NULL to skip saving residuals
PADJ_METHOD    <- "BH"
QV_MIN         <- 20        # Xenium transcript quality filter
ORG_DB_NAME    <- "org.Hs.eg.db"   # human (cervical); switch to org.Mm.eg.db for mouse

# Variant-E-specific knobs
INCLUDE_AREA_COV  <- TRUE        # Variant A: FALSE
WINSORIZE_FRAC    <- 0.005       # Variant A: NULL
PADJ_PER_METRIC   <- TRUE        # Variant A: FALSE
INCLUDE_ANISO_FRAC<- TRUE        # Variant A: FALSE
K_AREA_COV        <- 10          # Variant A: n/a

# ===== Optional analyses =====
RUN_LINEAR_REGRESSION <- FALSE   # set TRUE to run gSEM vs simple linear regression comparison (now redundant with gSEM)
FORCE_RERUN_RESIDUALS <- TRUE   # set TRUE to delete stale r_gene batch files + checkpoint before gSEM
                                 # Required when cell count changes (e.g. after adjusting MIN_CELL_AREA)

cat(sprintf("NCORES: %d | K_SPATIAL: %d | checkpoint every %d genes\n",
            NCORES, K_SPATIAL, CHECKPOINT_EVERY))

# ══════════════════════════════════════════════════════════════════════════════
# v2 EXTENDED — cell shape / morphology
# ══════════════════════════════════════════════════════════════════════════════
INCLUDE_SHAPE_METRICS <- TRUE     # FALSE reproduces a stock V5 production run

SHAPE_METRICS <- c("shape_index", "elongation", "circularity", "orient_alignment")

# Per-metric variance-stabilising transform applied BEFORE the spatial smooth.
# helper_functions.R::parallel_gsem_regression() hard-codes log() for every
# metric. That is correct for the strictly-positive mechanical metrics, but
# orient_alignment is exactly 0 whenever the stress axis and the cell long axis
# are orthogonal, and log(0) = -Inf turns that entire metric into NA.
# parallel_gsem_regression_ext() (defined below) honours this table instead.
METRIC_TRANSFORM <- list(
  pressure            = "log",    # > 0 after VMSI QC
  isotropic_stress    = "log",    # > 0 after VMSI QC
  anisotropic_stress  = "log",    # >= 0; winsorised away from 0
  anisotropy_fraction = "log",    # clamped to >= 1e-6 in the winsorise cell
  shape_index         = "log",    # p / sqrt(A) > 0
  elongation          = "log",    # inertia eigenvalue ratio > 0
  circularity         = "log",    # 4*pi*A / p^2, in (0, 1]
  orient_alignment    = "log1p"   # |cos(2*dtheta)| in [0, 1] -- CAN BE EXACTLY 0
)

SHAPE_WINSORIZE_FRAC <- WINSORIZE_FRAC  # NULL to leave shape metrics untouched
SHAPE_INDEX_CRITICAL <- 3.81            # vertex-model rigidity transition (Bi et al. 2015)

# Parity guard: proves parallel_gsem_regression_ext() reproduces the V5 helper
# exactly on the mechanical metrics. Set to 0 to skip (not recommended).
PARITY_CHECK_N_GENES <- 25
PARITY_TOL           <- 1e-9

GO_PADJ <- 0.1   # padj cutoff feeding the GO overrepresentation analysis

cat(sprintf("Shape metrics: %s\n",
            if (INCLUDE_SHAPE_METRICS) paste(SHAPE_METRICS, collapse = ", ") else "DISABLED"))


### Preprocessing

#### Library imports

In [ ]:
suppressWarnings(suppressMessages(library(sp)))
suppressWarnings(suppressMessages(library(mgcv)))
suppressWarnings(suppressMessages(library(dplyr)))
suppressWarnings(suppressMessages(library(progress)))
suppressWarnings(suppressMessages(library(parallel)))
suppressWarnings(suppressMessages(library(ggplot2)))
suppressWarnings(suppressMessages(library(ggpubr)))
suppressWarnings(suppressMessages(library(doParallel)))
suppressWarnings(suppressMessages(library(stringr)))
suppressWarnings(suppressMessages(library(BuenColors)))
suppressWarnings(suppressMessages(library(grid)))
suppressWarnings(suppressMessages(library(Matrix)))
suppressWarnings(suppressMessages(library(data.table)))
suppressWarnings(suppressMessages(library(EnhancedVolcano)))
suppressWarnings(suppressMessages(library(clusterProfiler)))
suppressWarnings(suppressMessages(library(FNN)))

suppressWarnings(suppressMessages(library(ORG_DB_NAME, character.only = TRUE)))
ORG_DB <- get(ORG_DB_NAME, envir = asNamespace(ORG_DB_NAME))
source("../../helper_functions.R")

#### Load and stitch tiles

For each tile in `TILE_NAMES`:

1. Load the per-cell mechanics CSV emitted by MB_02 (`{tile}_cell_results.csv`).
2. Convert tile-local centroids to global slide coordinates using the `Y/X` offsets parsed from the tile name.
3. Stream the tile's transcripts CSV, filter to assigned cells with `qv >= QV_MIN`, and aggregate to long-format `(global_cell_id, gene) -> count`.

After all tiles are read, build a single sparse `cells x genes` matrix (`Matrix::dgCMatrix`) once &mdash; never densify.

In [ ]:
PAD <- 10  # pixel border added during VMSI image preprocessing (matches run_spatial_regression_tile.R)

tensionmap_list <- vector("list", length(TILE_NAMES))
tx_long_list    <- vector("list", length(TILE_NAMES))

for (k in seq_along(TILE_NAMES)) {
  tp <- tile_paths(TILE_NAMES[k])
  cat(sprintf("[%d/%d] %s\n", k, length(TILE_NAMES), tp$tile))

  # ---- per-cell mechanics from MB_02 ----
  tm <- data.table::fread(file = tp$tensionmap)
  # VMSI is run on a 2x-upscaled image with a PAD-px border; convert to Xenium local px
  tm[, local_x := centroid_x / 2 - PAD]
  tm[, local_y := centroid_y / 2 - PAD]
  tm <- tm[is.finite(pressure)]
  if (nrow(tm) == 0L) { cat("  no valid VMSI cells; skipping\n"); next }

  # ---- transcripts: filter ----
  tx <- data.table::fread(file = tp$transcripts)
  tx <- tx[cell_id != "UNASSIGNED" & qv >= QV_MIN]
  tx <- tx[!grepl("^NegControl|^UnassignedCodeword|^BLANK|^DeprecatedCodeword", feature_name)]
  if (nrow(tx) == 0L) { cat("  no usable transcripts; skipping\n"); next }

  # Xenium cell centroids from transcript pixel coordinates
  xcol <- if ("x_local_px" %in% names(tx)) "x_local_px" else "x_location"
  ycol <- if ("y_local_px" %in% names(tx)) "y_local_px" else "y_location"
  xen_cents <- tx[, .(xen_x = mean(get(xcol), na.rm = TRUE),
                       xen_y = mean(get(ycol), na.rm = TRUE)), by = cell_id]

  # ---- nearest-neighbour match: each VMSI cell -> closest Xenium barcode ----
  # FNN::get.knnx uses a k-d tree: O((n+m) log n) vs O(n*m) for the naive loop.
  # Required when any tile has > ~100K cells; negligible overhead at smaller sizes.
  vmsi_xy <- as.matrix(tm[, .(local_x, local_y)])
  xen_xy  <- as.matrix(xen_cents[, .(xen_x, xen_y)])
  match_idx <- FNN::get.knnx(xen_xy, vmsi_xy, k = 1L)$nn.index[, 1L]

  tm[, xenium_id := xen_cents$cell_id[match_idx]]
  tm <- tm[!duplicated(xenium_id)]  # one VMSI cell per Xenium barcode

  # ---- global slide coordinates (add tile offset after local->px transform) ----
  tm[, centroid_x := local_x + tp$x_off]
  tm[, centroid_y := local_y + tp$y_off]
  tm[, cell_id    := paste0(tp$tile_id, "_", xenium_id)]
  tm[, tile_id    := tp$tile_id]
  tensionmap_list[[k]] <- tm

  # ---- long counts: remap to global cell_id, keep only matched cells ----
  tx[, cell_id := paste0(tp$tile_id, "_", cell_id)]
  tx <- tx[cell_id %in% tm$cell_id]
  tx_long_list[[k]] <- tx[, .(count = .N), by = .(cell_id, feature_name)]

  cat(sprintf("  VMSI: %d cells | Xenium: %d barcodes | matched: %d\n",
              nrow(tm), nrow(xen_cents), nrow(tm)))
}

# ---- deduplicate overlap zone: each cell owned by exactly one tile ----
all_rows <- as.integer(sub("^tile_R(\\d+)_C\\d+_.*", "\\1", TILE_NAMES))
all_cols <- as.integer(sub("^tile_R\\d+_C(\\d+)_.*", "\\1", TILE_NAMES))
max_row  <- max(all_rows)
max_col  <- max(all_cols)

for (k in seq_along(tensionmap_list)) {
  if (is.null(tensionmap_list[[k]])) next
  tp  <- tile_paths(TILE_NAMES[k])
  tm  <- tensionmap_list[[k]]
  x_max <- if (all_cols[k] == max_col) Inf else tp$x_off + TILE_PX - OVERLAP
  y_max <- if (all_rows[k] == max_row) Inf else tp$y_off + TILE_PX - OVERLAP
  keep  <- tm$centroid_x < x_max & tm$centroid_y < y_max
  tensionmap_list[[k]] <- tm[keep]
  tx_long_list[[k]]    <- tx_long_list[[k]][cell_id %in% tm$cell_id[keep]]
}

tensionmap_res <- data.table::rbindlist(tensionmap_list, use.names = TRUE, fill = TRUE)
tx_long        <- data.table::rbindlist(tx_long_list,    use.names = TRUE, fill = TRUE)
cat(sprintf("Stitched: %d cells, %d transcript rows across %d tile(s)\n",
            nrow(tensionmap_res), nrow(tx_long), length(TILE_NAMES)))

# ---- sparse cells x genes matrix, built once after all tiles are stitched ----
all_cells  <- unique(tx_long$cell_id)
all_genes  <- unique(tx_long$feature_name)
gex_sparse <- Matrix::sparseMatrix(
  i = match(tx_long$cell_id,      all_cells),
  j = match(tx_long$feature_name, all_genes),
  x = tx_long$count,
  dims = c(length(all_cells), length(all_genes)),
  dimnames = list(all_cells, all_genes)
)
rm(tx_long, tx_long_list, tensionmap_list); gc()
cat(sprintf("GEX matrix: %d cells x %d genes (%s sparse)\n",
            nrow(gex_sparse), ncol(gex_sparse),
            format(object.size(gex_sparse), units = "Mb")))

# ---- align: intersect tensionmap and GEX ----
common <- intersect(tensionmap_res$cell_id, rownames(gex_sparse))
tensionmap_res <- tensionmap_res[match(common, cell_id)]
gex_sparse     <- gex_sparse[common, , drop = FALSE]
tensionmap_res <- as.data.frame(tensionmap_res)
rownames(tensionmap_res) <- tensionmap_res$cell_id
cat(sprintf("After alignment: %d cells in both tables\n", length(common)))

In [ ]:
tensionmap_res[['isotropic_stress']] <- (tensionmap_res[['stresstensor_eigval1']] + tensionmap_res[['stresstensor_eigval2']]) / 2
tensionmap_res[['anisotropy_fraction']] <-
  (tensionmap_res[['stresstensor_eigval1']] - tensionmap_res[['stresstensor_eigval2']]) /
  (tensionmap_res[['stresstensor_eigval1']] + tensionmap_res[['stresstensor_eigval2']] + 1e-12)

#### v2 EXTENDED: derive cell shape / morphology metrics

Computed from the VMSI vertex-model geometry already present in
`{tile}_relabeled_cell_results.csv` — `perimeter`, `area`, and the inertia-tensor
eigenvalues/orientation. Definitions match the R3C14 gSEM QC trial
(`gSEM QC/run_MB05.4_fixed.R`), so v2 results are directly comparable to it.

A note on `elongation`: VMSI does not guarantee `I₁ ≥ I₂`, so the raw ratio can fall below 1.
The cell reports what fraction of cells that affects — a figure near 50% means the eigenvalues
come out unordered and the metric behaves as a folded aspect ratio. This is reported rather
than silently corrected, because the R3C14 trial used the raw ratio and changing it here would
break comparability with that run.

In [ ]:
# ── v2 EXTENDED: cell shape / morphology metrics ─────────────────────────────
#   shape_index       p / sqrt(A)         dimensionless fluidity parameter
#   elongation        I1 / I2             inertia-tensor aspect ratio
#   circularity       4*pi*A / p^2        1 = perfect circle
#   orient_alignment  |cos(2*dtheta)|     stress-vs-long-axis co-alignment
#
# shape_index and circularity are scale-invariant, so they are not collinear with
# the s(log_cell_area) covariate that INCLUDE_AREA_COV adds to every smooth.
#
# The shape metrics come from VMSI geometry, while log_cell_area comes from the
# Xenium cells.csv.gz segmentation (see the normalisation cell). The two
# segmentations are independent, so the covariate does not absorb the shape
# signal by construction -- the collinearity cell below checks this empirically.

if (INCLUDE_SHAPE_METRICS) {

  need <- c("perimeter", "area", "inertiatensor_eigval1", "inertiatensor_eigval2")
  miss <- setdiff(need, names(tensionmap_res))
  if (length(miss) > 0)
    stop(sprintf("VMSI output is missing required shape columns: %s",
                 paste(miss, collapse = ", ")))

  EPS <- 1e-20

  tensionmap_res[["shape_index"]] <-
    tensionmap_res[["perimeter"]] / sqrt(tensionmap_res[["area"]] + EPS)

  tensionmap_res[["elongation"]] <-
    tensionmap_res[["inertiatensor_eigval1"]] /
    (tensionmap_res[["inertiatensor_eigval2"]] + EPS)

  tensionmap_res[["circularity"]] <-
    (4 * pi * tensionmap_res[["area"]]) /
    (tensionmap_res[["perimeter"]]^2 + EPS)

  orient_cols <- c("stresstensor_orientation", "inertiatensor_orientation")
  if (all(orient_cols %in% names(tensionmap_res))) {
    tensionmap_res[["orient_alignment"]] <- abs(cos(2 * (
      tensionmap_res[["stresstensor_orientation"]] -
      tensionmap_res[["inertiatensor_orientation"]])))
  } else {
    warning("orientation columns absent from VMSI output - dropping orient_alignment")
    SHAPE_METRICS <- setdiff(SHAPE_METRICS, "orient_alignment")
  }

  cat("Shape metrics derived from", nrow(tensionmap_res), "cells\n\n")
  for (m in SHAPE_METRICS) {
    v  <- tensionmap_res[[m]]
    vf <- v[is.finite(v)]
    cat(sprintf("  %-17s finite=%7d  min=%9.4g  median=%9.4g  max=%9.4g  n<=0=%d\n",
                m, length(vf), min(vf), median(vf), max(vf),
                sum(vf <= 0)))
  }

  frac_lt1 <- mean(tensionmap_res[["elongation"]] < 1, na.rm = TRUE)
  cat(sprintf("\n  elongation < 1 in %.1f%% of cells", 100 * frac_lt1))
  cat(if (frac_lt1 > 0.35 && frac_lt1 < 0.65)
        "  <-- inertia eigenvalues appear UNORDERED; treat as folded aspect ratio\n" else "\n")

  cat(sprintf("  shape_index > %.2f (fluid-like) in %.1f%% of cells\n",
              SHAPE_INDEX_CRITICAL,
              100 * mean(tensionmap_res[["shape_index"]] > SHAPE_INDEX_CRITICAL,
                         na.rm = TRUE)))

} else {
  SHAPE_METRICS <- character(0)
  cat("INCLUDE_SHAPE_METRICS = FALSE - running stock V5 mechanical metrics only\n")
}


#### VMSI QC: remove mechanically unsolved cells

VMSI only solves pressure and stress for cells in `involved_cells` — the subset adjacent to bulk 3-way vertices where the force-balance equations are fully constrained. All other cells (typically those bordering the tissue edge, or cells removed due to degenerate vertex geometry) retain their default initialisation of `pressure = 0`, `stress = [0, 0, 0]`.

These zeros are not a physical result — they reflect cells that were never entered into the optimiser. They are excluded before regression because `log(0) = -Inf` would cause the GAM fits to fail. The filter is applied jointly to `tensionmap_res` and `gex_sparse` so the two remain in lock-step row alignment.

> **v2 extended:** the same filter additionally drops cells whose shape metrics are
> non-finite (degenerate VMSI polygons with zero area or perimeter) or non-positive where a
> `log` transform is used. Without this, a single bad polygon aborts the GAM fit for that
> metric across every gene.

In [ ]:
vmsi_valid <- tensionmap_res$pressure > 0 & tensionmap_res$isotropic_stress > 0
n_mech_bad <- sum(!vmsi_valid)

# ── v2 EXTENDED: shape metrics must be usable under their declared transform ──
n_shape_bad <- 0L
if (INCLUDE_SHAPE_METRICS && length(SHAPE_METRICS) > 0) {
  shape_ok <- rep(TRUE, nrow(tensionmap_res))
  for (m in SHAPE_METRICS) {
    v   <- tensionmap_res[[m]]
    tfm <- METRIC_TRANSFORM[[m]]
    ok  <- is.finite(v)
    if (identical(tfm, "log"))   ok <- ok & v >  0
    if (identical(tfm, "log1p")) ok <- ok & v > -1
    if (sum(!ok) > 0)
      cat(sprintf("  %-17s %d cell(s) unusable under transform '%s'\n", m, sum(!ok), tfm))
    shape_ok <- shape_ok & ok
  }
  n_shape_bad <- sum(!shape_ok & vmsi_valid)   # attributable to shape alone
  vmsi_valid  <- vmsi_valid & shape_ok
}

n_dropped <- sum(!vmsi_valid)

tensionmap_res <- tensionmap_res[vmsi_valid, ]
gex_sparse     <- gex_sparse[vmsi_valid, , drop = FALSE]

cat(sprintf("VMSI QC: dropped %d cells total\n", n_dropped))
cat(sprintf("         %d mechanically degenerate (pressure=0 or stress_mag=0)\n", n_mech_bad))
cat(sprintf("         %d additionally dropped on shape-metric validity\n", n_shape_bad))
cat(sprintf("         %d cells remain\n", nrow(tensionmap_res)))


In [ ]:
head(tensionmap_res)

In [ ]:
# ── Normalisation ─────────────────────────────────────────────────────────────
# Normalise gex_sparse in place so all downstream cells receive normalised values.
# Primary:  Xenium cell_area from cells.csv.gz — the area Xenium used when
#           assigning transcripts, so it is the correct denominator.
#           VMSI area (tensionmap_res$area) is NOT used: VMSI segments a
#           different image independently and its boundaries don't match Xenium.
# Fallback: library-size CPM when cells.csv.gz is absent.
# log1p is applied only to stored @x values; log1p(0)=0 so zeros stay zero.

cells_csv <- file.path(XENIUM_DIR, "cells.csv.gz")

if (NORM_METHOD == "area" && file.exists(cells_csv)) {
  xen_cells <- data.table::fread(file = cells_csv,
                                  select = c("cell_id", "cell_area"))
  # Strip tile_id prefix from rownames to recover bare Xenium barcodes.
  # Robust to tile_id formats.
  tile_ids <- unique(tensionmap_res$tile_id)
  bare_ids <- rownames(gex_sparse)
  for (tid in tile_ids) {
    bare_ids <- sub(paste0("^", tid, "_"), "", bare_ids)
  }
  cell_areas <- xen_cells$cell_area[match(bare_ids, xen_cells$cell_id)]

  n_missing <- sum(is.na(cell_areas))
  if (n_missing > 0)
    warning(sprintf("%d cells had no matching area in cells.csv.gz and will be dropped", n_missing))
  keep <- !is.na(cell_areas)
  gex_sparse     <- gex_sparse[keep, , drop = FALSE]
  tensionmap_res <- tensionmap_res[keep, ]
  cell_areas     <- cell_areas[keep]

  area_floor <- if (is.null(MIN_CELL_AREA)) quantile(cell_areas, 0.01) else MIN_CELL_AREA
  keep2      <- cell_areas >= area_floor
  if (any(!keep2)) {
    cat(sprintf("  Dropping %d cells with area < %.1f µm²\n", sum(!keep2), area_floor))
    gex_sparse     <- gex_sparse[keep2, , drop = FALSE]
    tensionmap_res <- tensionmap_res[keep2, ]
    cell_areas     <- cell_areas[keep2]
  }

  # Store on tensionmap_res so the area covariate (INCLUDE_AREA_COV) is available
  # to parallel_gsem_regression, which auto-derives log_cell_area from this column.
  tensionmap_res$cell_area <- cell_areas

  size_factors <- cell_areas / median(cell_areas)
  cat(sprintf("Normalising by Xenium cell_area (n=%d, median=%.1f µm²)\n",
              length(cell_areas), median(cell_areas)))

} else {
  if (NORM_METHOD == "area")
    warning("cells.csv.gz not found — falling back to library-size (CPM) normalisation")
  lib_sizes    <- Matrix::rowSums(gex_sparse)
  size_factors <- lib_sizes / 10000
  cat(sprintf("Normalising by library size (CPM, n=%d cells)\n", nrow(gex_sparse)))
}

# Diagonal %*% sparse drops rownames — save and restore them
.cell_ids     <- rownames(gex_sparse)
gex_sparse    <- Matrix::Diagonal(x = 1 / size_factors) %*% gex_sparse
rownames(gex_sparse) <- .cell_ids
gex_sparse@x  <- log1p(gex_sparse@x)
cat(sprintf("Normalised gex_sparse: %d cells x %d genes\n",
            nrow(gex_sparse), ncol(gex_sparse)))

In [ ]:
.winsorize <- function(x, p) {
  qs <- quantile(x, c(p, 1 - p), na.rm = TRUE)
  pmin(pmax(x, qs[1]), qs[2])
}

if (!is.null(WINSORIZE_FRAC)) {
  cat(sprintf("Winsorizing stress metrics at %.1f%% / %.1f%% tails\n",
              100 * WINSORIZE_FRAC, 100 * (1 - WINSORIZE_FRAC)))
  for (m in c("pressure", "isotropic_stress", "anisotropic_stress")) {
    if (m %in% names(tensionmap_res)) {
      tensionmap_res[[m]] <- .winsorize(tensionmap_res[[m]], WINSORIZE_FRAC)
    }
  }
  if (INCLUDE_ANISO_FRAC && "anisotropy_fraction" %in% names(tensionmap_res)) {
    tensionmap_res[["anisotropy_fraction"]] <-
      pmax(tensionmap_res[["anisotropy_fraction"]], 1e-6)
    tensionmap_res[["anisotropy_fraction"]] <-
      .winsorize(tensionmap_res[["anisotropy_fraction"]], WINSORIZE_FRAC)
  }
}

# ── v2 EXTENDED: winsorise shape metrics on the same schedule ────────────────
# elongation in particular has a long right tail from sliver polygons; without
# this a handful of cells dominate the log-scale spatial smooth.
# orient_alignment is left with its zeros intact -- they are physically
# meaningful (stress orthogonal to the long axis), which is exactly why it is
# mapped to log1p rather than log.
if (INCLUDE_SHAPE_METRICS && !is.null(SHAPE_WINSORIZE_FRAC) && length(SHAPE_METRICS) > 0) {
  cat(sprintf("Winsorizing shape metrics at %.1f%% / %.1f%% tails\n",
              100 * SHAPE_WINSORIZE_FRAC, 100 * (1 - SHAPE_WINSORIZE_FRAC)))
  for (m in setdiff(SHAPE_METRICS, "orient_alignment")) {
    before <- tensionmap_res[[m]]
    tensionmap_res[[m]] <- .winsorize(before, SHAPE_WINSORIZE_FRAC)
    if (identical(METRIC_TRANSFORM[[m]], "log"))
      tensionmap_res[[m]] <- pmax(tensionmap_res[[m]], 1e-12)
    cat(sprintf("  %-17s [%.4g, %.4g] -> [%.4g, %.4g]\n", m,
                min(before), max(before),
                min(tensionmap_res[[m]]), max(tensionmap_res[[m]])))
  }
}


#### v2 EXTENDED: shape metric QC

Sanity-check the distributions before they enter the model. The dashed line on `shape_index`
marks the vertex-model rigidity transition at ≈ 3.81 — tissue predominantly above it is
fluid-like/unjammed, below it solid-like/jammed. A distribution sitting entirely on one side
of that line is a signal that the shape metrics will have little dynamic range to work with
on this sample.

In [ ]:
if (INCLUDE_SHAPE_METRICS && length(SHAPE_METRICS) > 0) {
  dir.create(OUTPUT_DIR, recursive = TRUE, showWarnings = FALSE)

  shape_long <- do.call(rbind, lapply(SHAPE_METRICS, function(m)
    data.frame(metric = m, value = tensionmap_res[[m]], stringsAsFactors = FALSE)))
  shape_long <- shape_long[is.finite(shape_long$value), ]

  p_shape <- ggplot(shape_long, aes(x = value)) +
    geom_histogram(bins = 60, fill = "steelblue", colour = NA) +
    facet_wrap(~metric, scales = "free", ncol = 2) +
    labs(x = NULL, y = "Cells",
         title = sprintf("Cell shape metric distributions (%d cells)",
                         nrow(tensionmap_res))) +
    theme_bw() + theme(plot.title = element_text(hjust = 0.5))

  if ("shape_index" %in% SHAPE_METRICS) {
    p_shape <- p_shape + geom_vline(
      data = data.frame(metric = "shape_index", xi = SHAPE_INDEX_CRITICAL),
      aes(xintercept = xi), linetype = "dashed", colour = "red", linewidth = 0.4)
  }

  ggsave(file.path(OUTPUT_DIR, paste0(SAMPLE_NAME, "_shape_distributions.png")),
         p_shape, width = 9, height = 6, dpi = 150)
  print(p_shape)

  # Spatial map of shape_index -- confirms there IS spatial structure for the
  # GAM to remove, and that it is not a segmentation artefact tracking tile edges.
  if ("shape_index" %in% SHAPE_METRICS) {
    p_map <- ggplot(tensionmap_res,
                    aes(x = centroid_x, y = centroid_y, colour = shape_index)) +
      geom_point(size = 0.25, alpha = 0.7) +
      scale_colour_viridis_c(name = "shape index",
                             limits = quantile(tensionmap_res$shape_index,
                                               c(0.01, 0.99), na.rm = TRUE),
                             oob = scales::squish) +
      coord_equal() + theme_void() +
      ggtitle("Spatial distribution of cell shape index") +
      theme(plot.title = element_text(hjust = 0.5))
    ggsave(file.path(OUTPUT_DIR, paste0(SAMPLE_NAME, "_shape_index_map.png")),
           p_map, width = 8, height = 7, dpi = 150)
    print(p_map)
  }
}


### Perform spatial regression

Note: this step is computationally intensive as it takes ~10s to construct a model for each gene (tested on M1 Macbook Pro). For demonstration purposes we only do regression on a random subset of 100 genes.

In [ ]:
expr_frac       <- Matrix::colSums(gex_sparse > 0) / nrow(gex_sparse)
expressed_genes <- colnames(gex_sparse)[expr_frac >= EXPR_FRAC_MIN]
cat(sprintf("Expressed genes (>= %.0f%% of cells): %d / %d\n",
            100 * EXPR_FRAC_MIN, length(expressed_genes), ncol(gex_sparse)))

Set `N_GENES_SAMPLE` to an integer in the config cell for a trial run on a random subset, or leave it `NULL` to run gSEM on all expressed genes.

In [ ]:
set.seed(RANDOM_SEED)
genes_use <- if (!is.null(N_GENES_SAMPLE) && N_GENES_SAMPLE < length(expressed_genes)) {
  sample(expressed_genes, N_GENES_SAMPLE, replace = FALSE)
} else {
  expressed_genes
}
cat(sprintf("Genes for gSEM: %d\n", length(genes_use)))

### v2 EXTENDED: per-metric transform layer

`helper_functions.R::parallel_gsem_regression()` applies `log()` unconditionally to every
metric:

```r
r_metrics <- lapply(setNames(metrics, metrics), function(m) {
  vals <- log(tensionmap_res[[m]])          # <-- hard-coded
  vals - .fit_spatial(vals, ...)
})
```

That is correct for the four strictly-positive mechanical metrics, but `orient_alignment`
is **exactly 0** whenever the stress axis and the cell long axis are orthogonal, and
`log(0) = -Inf` propagates through the GAM to give an all-`NA` result for that metric.

`parallel_gsem_regression_ext()` below is a fork of the V5 function with exactly **one**
behavioural change — the transform is looked up per metric in `METRIC_TRANSFORM` rather than
hard-coded. Everything else (fork cluster, checkpointing, residual persistence, area
covariate, error handling) is unchanged. With every metric mapped to `"log"` it is
numerically identical to the V5 function; the parity cell asserts this on real data before
the full run starts.

> **TODO (pipeline):** fold `metric_transform` into `helper_functions.R` for V6 so this fork
> can be deleted and dataset notebooks stop carrying a copy of the driver.

In [ ]:
.apply_metric_transform <- function(x, tfm) {
  switch(tfm,
         log      = log(x),
         log1p    = log1p(x),
         identity = x,
         stop(sprintf("unknown metric transform: '%s'", tfm)))
}

# Fork of helper_functions.R::parallel_gsem_regression().
# ONLY difference: the metric transform is per-metric (metric_transform) rather
# than a hard-coded log(). Reuses .fit_spatial() from the sourced helper so the
# smoothing behaviour cannot drift from V5.
parallel_gsem_regression_ext <- function(genes, tensionmap_res, gex_res,
                                          k_sp = 300, model_fx = FALSE,
                                          method = 'fREML', ncores = 6,
                                          use_bam = TRUE,
                                          metrics = c('pressure',
                                                      'anisotropic_stress',
                                                      'isotropic_stress'),
                                          metric_transform = NULL,
                                          include_area_cov = FALSE,
                                          k_area = 10,
                                          checkpoint_file = NULL,
                                          checkpoint_every = 100L,
                                          residuals_dir = NULL) {

  if (is.null(metric_transform))
    metric_transform <- setNames(as.list(rep("log", length(metrics))), metrics)

  if (include_area_cov && !"log_cell_area" %in% names(tensionmap_res)) {
    if ("cell_area" %in% names(tensionmap_res)) {
      tensionmap_res$log_cell_area <- log(tensionmap_res$cell_area)
    } else {
      stop("include_area_cov=TRUE requires 'log_cell_area' or 'cell_area' in tensionmap_res")
    }
  }

  # Metric spatial residuals, computed once for all genes.
  r_metrics <- lapply(setNames(metrics, metrics), function(m) {
    tfm <- metric_transform[[m]]
    if (is.null(tfm))
      stop(sprintf("no transform defined for metric '%s'", m))
    vals <- .apply_metric_transform(tensionmap_res[[m]], tfm)
    if (!all(is.finite(vals)))
      stop(sprintf("metric '%s' has %d non-finite value(s) after transform '%s'",
                   m, sum(!is.finite(vals)), tfm))
    vals - .fit_spatial(vals, tensionmap_res, k_sp, model_fx, method, use_bam,
                        include_area_cov, k_area)
  })

  if (!is.null(residuals_dir)) {
    dir.create(residuals_dir, recursive = TRUE, showWarnings = FALSE)
    saveRDS(r_metrics, file.path(residuals_dir, "r_metrics.rds"))
    writeLines(rownames(tensionmap_res), file.path(residuals_dir, "cell_order.txt"))
    saveRDS(metric_transform[metrics],
            file.path(residuals_dir, "metric_transform.rds"))
    cat(sprintf("r_metrics saved to %s\n", residuals_dir))
  }

  genes_done <- character(0)
  if (!is.null(checkpoint_file) && file.exists(checkpoint_file)) {
    prev <- data.table::fread(checkpoint_file, showProgress = FALSE)
    genes_done <- unique(prev[["gene"]])
    cat(sprintf("Resuming: %d/%d genes already done\n", length(genes_done), length(genes)))
  }
  genes_remaining <- setdiff(genes, genes_done)

  batch_idx <- split(seq_along(genes_remaining),
                     ceiling(seq_along(genes_remaining) / checkpoint_every))

  tryCatch(doParallel::stopImplicitCluster(), error = function(e) NULL)
  cl <- parallel::makeForkCluster(ncores)
  doParallel::registerDoParallel(cl)
  on.exit({ parallel::stopCluster(cl); doParallel::stopImplicitCluster() }, add = TRUE)

  save_res <- !is.null(residuals_dir)

  all_results <- vector("list", length(batch_idx))
  for (b in seq_along(batch_idx)) {
    batch_genes <- genes_remaining[batch_idx[[b]]]

    if (save_res) {
      batch_list <- foreach(i = icount(length(batch_genes)),
                            .packages = 'mgcv',
                            .export = c('.fit_spatial')) %dopar% {
        gene <- batch_genes[i]
        res <- data.frame(gene=character(0), metric=character(0), beta=numeric(0),
                          pval=numeric(0), stat=numeric(0), error_msg=character(0),
                          stringsAsFactors=FALSE)
        r_Y_out <- NULL
        tryCatch({
          Y   <- as.numeric(t(gex_res[gene, rownames(tensionmap_res)]))
          r_Y <- Y - .fit_spatial(Y, tensionmap_res, k_sp, model_fx, method,
                                  use_bam, include_area_cov, k_area)
          r_Y_out <- r_Y
          for (m in names(r_metrics)) {
            r_X <- r_metrics[[m]]
            mod <- lm(r_Y ~ r_X)
            res <- rbind(res, data.frame(
              gene=gene, metric=m,
              beta=mod$coefficients[2],
              pval=summary(mod)$coefficients[2, 4],
              stat=summary(mod)$coefficients[2, 3],
              error_msg=NA_character_,
              stringsAsFactors=FALSE))
          }
        }, error = function(e) {
          for (m in names(r_metrics))
            res <<- rbind(res, data.frame(
              gene=gene, metric=m, beta=NA_real_, pval=NA_real_, stat=NA_real_,
              error_msg=conditionMessage(e), stringsAsFactors=FALSE))
        })
        list(stats = res, r_Y = r_Y_out)
      }

      batch_res <- do.call(rbind, lapply(batch_list, `[[`, "stats"))
      batch_r_Y <- do.call(cbind,  lapply(batch_list, `[[`, "r_Y"))
      colnames(batch_r_Y) <- batch_genes
      saveRDS(batch_r_Y,
              file.path(residuals_dir, sprintf("r_gene_batch_%04d.rds", b)))
      rm(batch_r_Y, batch_list); gc()

    } else {
      batch_res <- foreach(i = icount(length(batch_genes)), .combine = rbind,
                           .packages = 'mgcv',
                           .export = c('.fit_spatial')) %dopar% {
        gene <- batch_genes[i]
        res <- data.frame(gene=character(0), metric=character(0), beta=numeric(0),
                          pval=numeric(0), stat=numeric(0), error_msg=character(0),
                          stringsAsFactors=FALSE)
        for (m in names(r_metrics)) {
          tryCatch({
            Y   <- as.numeric(t(gex_res[gene, rownames(tensionmap_res)]))
            r_Y <- Y - .fit_spatial(Y, tensionmap_res, k_sp, model_fx, method,
                                    use_bam, include_area_cov, k_area)
            r_X <- r_metrics[[m]]
            mod <- lm(r_Y ~ r_X)
            res <- rbind(res, data.frame(
              gene=gene, metric=m,
              beta=mod$coefficients[2],
              pval=summary(mod)$coefficients[2, 4],
              stat=summary(mod)$coefficients[2, 3],
              error_msg=NA_character_,
              stringsAsFactors=FALSE))
          }, error = function(e) {
            res <<- rbind(res, data.frame(
              gene=gene, metric=m, beta=NA_real_, pval=NA_real_, stat=NA_real_,
              error_msg=conditionMessage(e), stringsAsFactors=FALSE))
          })
        }
        res
      }
    }

    all_results[[b]] <- batch_res
    if (!is.null(checkpoint_file)) {
      data.table::fwrite(batch_res, checkpoint_file,
                         append = file.exists(checkpoint_file))
      cat(sprintf("  Batch %d/%d done (%d genes); checkpoint written\n",
                  b, length(batch_idx), length(batch_genes)))
    }
  }

  reg_res <- do.call(rbind, all_results)
  if (length(genes_done) > 0L && !is.null(checkpoint_file)) {
    prev      <- data.table::fread(checkpoint_file, showProgress = FALSE)
    prev_done <- prev[prev[["gene"]] %in% genes_done, ]
    reg_res   <- rbind(as.data.frame(prev_done), reg_res)
  }
  reg_res
}

cat("parallel_gsem_regression_ext() defined\n")


#### Assemble the metric list and validate transforms

Every metric must have a `METRIC_TRANSFORM` entry, a column in `tensionmap_res`, and values
compatible with its transform. This cell fails loudly here rather than letting a metric come
back as a column of `NA` after a multi-hour run.

In [ ]:
metrics_mech <- c('pressure', 'anisotropic_stress', 'isotropic_stress')
if (INCLUDE_ANISO_FRAC) metrics_mech <- c(metrics_mech, 'anisotropy_fraction')

metrics_use <- metrics_mech
if (INCLUDE_SHAPE_METRICS && length(SHAPE_METRICS) > 0)
  metrics_use <- c(metrics_use, SHAPE_METRICS)

for (m in metrics_use) {
  tfm <- METRIC_TRANSFORM[[m]]
  if (is.null(tfm))
    stop(sprintf("METRIC_TRANSFORM has no entry for metric '%s'", m))
  if (!m %in% names(tensionmap_res))
    stop(sprintf("tensionmap_res has no column '%s'", m))

  v   <- tensionmap_res[[m]]
  bad <- switch(tfm,
                log      = sum(!is.finite(v) | v <= 0),
                log1p    = sum(!is.finite(v) | v <= -1),
                identity = sum(!is.finite(v)))
  cat(sprintf("  %-20s transform=%-8s bad=%d\n", m, tfm, bad))
  if (bad > 0)
    stop(sprintf("metric '%s' has %d value(s) incompatible with transform '%s'",
                 m, bad, tfm))
}

cat(sprintf("\ngSEM will fit %d metric(s) x %d gene(s) on %d cells\n",
            length(metrics_use), length(genes_use), nrow(tensionmap_res)))
cat(sprintf("  mechanical: %s\n", paste(metrics_mech, collapse = ", ")))
cat(sprintf("  shape     : %s\n",
            if (length(setdiff(metrics_use, metrics_mech)))
              paste(setdiff(metrics_use, metrics_mech), collapse = ", ") else "(none)"))


#### Collinearity between shape and mechanical metrics

Shape and stress are physically coupled, so overlapping gene hits are expected — the question
is whether shape carries *independent* information. Two checks:

- **Raw (transformed) correlation** — how coupled the metrics are before spatial detrending.
- **Correlation with `log_cell_area`** — confirms `shape_index` and `circularity` really are
  scale-invariant and so are not absorbed by the `s(log_cell_area)` covariate.

The correlation that actually governs interpretation is between the *spatial residuals*; that
one is computed after the run, once `r_metrics.rds` exists.

In [ ]:
mm <- sapply(metrics_use, function(m)
  .apply_metric_transform(tensionmap_res[[m]], METRIC_TRANSFORM[[m]]))
colnames(mm) <- metrics_use

# The normalisation cell attaches cell_area (not log_cell_area) to tensionmap_res;
# parallel_gsem_regression* derives the log internally. Derive it here too so the
# scale-invariance check below can run before the model is fitted.
if (!"log_cell_area" %in% names(tensionmap_res) &&
     "cell_area"     %in% names(tensionmap_res)) {
  tensionmap_res$log_cell_area <- log(tensionmap_res$cell_area)
}
if ("log_cell_area" %in% names(tensionmap_res))
  mm <- cbind(mm, log_cell_area = tensionmap_res$log_cell_area)

zero_var <- apply(mm, 2, function(x) length(unique(x[is.finite(x)])) < 2)
if (any(zero_var)) {
  warning(sprintf("dropping zero-variance column(s) from collinearity check: %s",
                  paste(colnames(mm)[zero_var], collapse = ", ")))
  mm <- mm[, !zero_var, drop = FALSE]
}

cmat <- cor(mm, method = "spearman", use = "complete.obs")

cm_long <- expand.grid(a = rownames(cmat), b = colnames(cmat),
                       stringsAsFactors = FALSE)
cm_long$rho <- as.vector(cmat)

p_coll <- ggplot(cm_long, aes(x = a, y = b, fill = rho)) +
  geom_tile(colour = "white") +
  geom_text(aes(label = sprintf("%.2f", rho)), size = 2.8) +
  scale_fill_gradient2(low = "#2166AC", mid = "white", high = "#B2182B",
                       midpoint = 0, limits = c(-1, 1), name = "Spearman") +
  labs(x = NULL, y = NULL,
       title = "Metric collinearity (transformed scale, pre-detrending)") +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        plot.title  = element_text(hjust = 0.5))

dir.create(OUTPUT_DIR, recursive = TRUE, showWarnings = FALSE)
ggsave(file.path(OUTPUT_DIR, paste0(SAMPLE_NAME, "_metric_collinearity.png")),
       p_coll, width = 8, height = 7, dpi = 150)
write.csv(cmat, file.path(OUTPUT_DIR, paste0(SAMPLE_NAME, "_metric_collinearity.csv")))
print(p_coll)

if ("log_cell_area" %in% colnames(cmat)) {
  cat("\nCorrelation with log_cell_area (scale-invariant metrics should be ~0):\n")
  for (m in intersect(metrics_use, rownames(cmat)))
    cat(sprintf("  %-20s rho = %+.3f\n", m, cmat[m, "log_cell_area"]))
}
rm(mm); gc()


In [ ]:
if (FORCE_RERUN_RESIDUALS) {
  cat("FORCE_RERUN_RESIDUALS = TRUE — clearing stale residuals before gSEM run\n")

  # Remove any existing r_gene batch files (may be from a different-sized cell run)
  batch_to_del <- list.files(RESIDUALS_DIR, pattern = "r_gene_batch_", full.names = TRUE)
  if (length(batch_to_del) > 0) {
    ok <- file.remove(batch_to_del)
    cat(sprintf("  Deleted %d / %d batch file(s)\n", sum(ok), length(batch_to_del)))
  } else {
    cat("  No batch files found — nothing to delete\n")
  }

  # Remove checkpoint so gSEM re-runs all genes rather than skipping them as 'done'
  if (file.exists(CHECKPOINT_FILE)) {
    file.remove(CHECKPOINT_FILE)
    cat(sprintf("  Deleted checkpoint: %s\n", CHECKPOINT_FILE))
  }

  cat("Ready for a clean rerun. r_metrics.rds and cell_order.txt will be overwritten by gSEM.\n")
} else {
  existing <- list.files(RESIDUALS_DIR, pattern = "r_gene_batch_", full.names = TRUE)
  cat(sprintf("FORCE_RERUN_RESIDUALS = FALSE — %d existing batch file(s) will be preserved by checkpoint\n",
              length(existing)))
  if (length(existing) > 0)
    cat("  Tip: set FORCE_RERUN_RESIDUALS <- TRUE if cell count changed since last run\n")
}

In [ ]:
gex_res_dense <- as.data.frame(t(as.matrix(gex_sparse[, genes_use, drop = FALSE])))
rm(gex_sparse); gc()  # sparse no longer needed; recover memory before fork workers start

dir.create(OUTPUT_DIR, recursive = TRUE, showWarnings = FALSE)
cat(sprintf("Dense GEX for gSEM: %d genes x %d cells (%s)\n",
            nrow(gex_res_dense), ncol(gex_res_dense),
            format(object.size(gex_res_dense), units = "Gb")))


#### Parity check — the ext driver must reproduce V5 exactly

Runs both `parallel_gsem_regression()` (V5, sourced from `helper_functions.R`) and
`parallel_gsem_regression_ext()` (this notebook) over a small gene subset, restricted to the
mechanical metrics where both apply `log`. Betas and p-values must agree to `PARITY_TOL`.

This is what makes v2 results comparable with the 13 existing MechanoMap production runs: if
this cell passes, the mechanical columns of the output are the same numbers the stock
pipeline would have produced, and the shape columns are pure addition.

In [ ]:
if (PARITY_CHECK_N_GENES > 0) {

  parity_genes <- head(genes_use, min(PARITY_CHECK_N_GENES, length(genes_use)))
  cat(sprintf("Parity check: %d genes x %d mechanical metrics\n",
              length(parity_genes), length(metrics_mech)))

  ref <- parallel_gsem_regression(
    parity_genes, tensionmap_res, gex_res_dense,
    k_sp = K_SPATIAL, model_fx = MODEL_FX, ncores = NCORES, use_bam = USE_BAM,
    method = gSEM_METHOD, metrics = metrics_mech,
    include_area_cov = INCLUDE_AREA_COV, k_area = K_AREA_COV,
    checkpoint_file = NULL, residuals_dir = NULL)

  ext <- parallel_gsem_regression_ext(
    parity_genes, tensionmap_res, gex_res_dense,
    k_sp = K_SPATIAL, model_fx = MODEL_FX, ncores = NCORES, use_bam = USE_BAM,
    method = gSEM_METHOD, metrics = metrics_mech,
    metric_transform = METRIC_TRANSFORM,
    include_area_cov = INCLUDE_AREA_COV, k_area = K_AREA_COV,
    checkpoint_file = NULL, residuals_dir = NULL)

  cmp <- merge(ref[, c("gene", "metric", "beta", "pval")],
               ext[, c("gene", "metric", "beta", "pval")],
               by = c("gene", "metric"), suffixes = c("_ref", "_ext"))

  if (nrow(cmp) == 0)
    stop("PARITY FAILED - no overlapping rows between the two drivers")

  d_beta <- max(abs(cmp$beta_ref - cmp$beta_ext), na.rm = TRUE)
  d_pval <- max(abs(cmp$pval_ref - cmp$pval_ext), na.rm = TRUE)

  cat(sprintf("  rows compared : %d\n", nrow(cmp)))
  cat(sprintf("  max |d beta|  : %.3g\n", d_beta))
  cat(sprintf("  max |d pval|  : %.3g\n", d_pval))

  if (!(is.finite(d_beta) && is.finite(d_pval) &&
        d_beta < PARITY_TOL && d_pval < PARITY_TOL))
    stop("PARITY FAILED - ext driver diverges from the V5 helper on log metrics. ",
         "Do not trust the mechanical results from this run.")

  cat("\n  PASS - ext driver reproduces the V5 gSEM path exactly.\n")
  rm(ref, ext, cmp); gc()

} else {
  cat("Parity check SKIPPED (PARITY_CHECK_N_GENES = 0)\n")
}


In [ ]:
# isotropic_stress = (eigval1 + eigval2) / 2; log-residuals are equivalent to those from
# stresstensor_magnitude after spatial smoothing -- using isotropic_stress directly for
# physical interpretability.
gsem_reg_res_smooth <- parallel_gsem_regression_ext(
  genes_use, tensionmap_res, gex_res_dense,
  k_sp = K_SPATIAL, model_fx = MODEL_FX, ncores = NCORES, use_bam = USE_BAM,
  method = gSEM_METHOD,
  metrics = metrics_use,
  metric_transform = METRIC_TRANSFORM,
  include_area_cov = INCLUDE_AREA_COV,
  k_area = K_AREA_COV,
  checkpoint_file = CHECKPOINT_FILE,
  checkpoint_every = CHECKPOINT_EVERY,
  residuals_dir = RESIDUALS_DIR
)

print("gSEM complete")


In [ ]:
# Check distribution of p-values
hist(gsem_reg_res_smooth$pval, breaks=100)

In [ ]:
# # DEBUG: Check zero-stress cells
# # Where do zero-stress cells sit spatially?
# zero_stress <- tensionmap_res[tensionmap_res$isotropic_stress == 0, ]
# cat("Zero-stress cells:", nrow(zero_stress), "\n")

# # Plot their positions — are they at tile edges?
# library(ggplot2)
# ggplot(tensionmap_res, aes(centroid_x, centroid_y,
#        color = isotropic_stress == 0)) +
#   geom_point(size = 0.5) +
#   scale_color_manual(values = c("grey80", "red")) +
#   theme_void() + ggtitle("Zero isotropic_stress cells")


In [ ]:
if (PADJ_PER_METRIC) {
  gsem_reg_res_smooth$padj <- NA_real_
  for (m in unique(gsem_reg_res_smooth$metric)) {
    idx <- gsem_reg_res_smooth$metric == m
    gsem_reg_res_smooth$padj[idx] <- p.adjust(
      gsem_reg_res_smooth$pval[idx], method = PADJ_METHOD)
  }
} else {
  gsem_reg_res_smooth$padj <- p.adjust(gsem_reg_res_smooth$pval,
                                        method = PADJ_METHOD)
}

In [ ]:
dir.create(OUTPUT_DIR, recursive = TRUE, showWarnings = FALSE)

results_csv <- file.path(OUTPUT_DIR, paste0(SAMPLE_NAME, "_v2ext_gsem_results.csv"))
write.csv(gsem_reg_res_smooth, results_csv, row.names = FALSE)

writeLines(TILE_NAMES, file.path(OUTPUT_DIR, paste0(SAMPLE_NAME, "_tiles_used.txt")))

n_sig  <- sum(gsem_reg_res_smooth$padj < 0.05 &
              is.na(gsem_reg_res_smooth$error_msg), na.rm = TRUE)
n_fail <- sum(!is.na(gsem_reg_res_smooth$error_msg))
cat(sprintf("%d tile(s) | %d cells | %d genes fit | %d significant (padj<0.05) | %d failed\n",
            length(TILE_NAMES), nrow(tensionmap_res), length(genes_use), n_sig, n_fail))

# ── v2 EXTENDED: per-metric breakdown, mechanical vs shape ───────────────────
summary_tbl <- do.call(rbind, lapply(metrics_use, function(m) {
  d <- gsem_reg_res_smooth[gsem_reg_res_smooth$metric == m, ]
  s <- d[!is.na(d$padj) & d$padj < 0.05, ]
  data.frame(
    metric    = m,
    class     = if (m %in% metrics_mech) "mechanical" else "shape",
    transform = METRIC_TRANSFORM[[m]],
    n_genes   = nrow(d),
    n_sig     = nrow(s),
    n_up      = sum(s$beta > 0, na.rm = TRUE),
    n_down    = sum(s$beta < 0, na.rm = TRUE),
    n_failed  = sum(!is.na(d$error_msg)),
    min_padj  = if (nrow(d)) min(d$padj, na.rm = TRUE) else NA_real_,
    stringsAsFactors = FALSE)
}))

write.csv(summary_tbl,
          file.path(OUTPUT_DIR, paste0(SAMPLE_NAME, "_v2ext_metric_summary.csv")),
          row.names = FALSE)
print(summary_tbl)
cat("\nResults written to:", results_csv, "\n")


### Further analysis of results

#### Generate volcano plots
Note: for this analysis the adjusted p-value threshold has been set to 0.1 for illustrative purposes. The threshold value used for the full analysis is 0.05.

In [ ]:
volcano_input <- gsem_reg_res_smooth

N_LABEL_PER_SIDE <- 15  # top-N UP + top-N DOWN labelled per volcano

for (curr_metric in unique(volcano_input$metric)) {
  metric_reg_res <- dplyr::filter(volcano_input, metric == curr_metric)
  metric_reg_res <- metric_reg_res[!is.na(metric_reg_res$padj), ]

  # Balanced label selection: top-N UP + top-N DOWN by padj.
  # Without this, one direction (often positive for pressure) consumes all label slots.
  sig <- metric_reg_res[metric_reg_res$padj < 0.05, ]
  top_up <- if (any(sig$beta > 0))
    sig[sig$beta > 0, ] %>% dplyr::arrange(padj) %>% head(N_LABEL_PER_SIDE) else
    data.frame(gene = character(0))
  top_dn <- if (any(sig$beta < 0))
    sig[sig$beta < 0, ] %>% dplyr::arrange(padj) %>% head(N_LABEL_PER_SIDE) else
    data.frame(gene = character(0))
  select_lab <- c(top_up$gene, top_dn$gene)

  points_cols <- ifelse(metric_reg_res$padj < 0.05, 'red', 'lightgrey')
  names(points_cols) <- metric_reg_res$gene

  # Floor padj at smallest representable positive double before ylim computation.
  # Without this, padj=0 (double underflow for extremely significant genes) gives
  # -log10(0)=Inf -> ylim=c(0,Inf) -> all finite points collapse to y~0.
  max_y <- max(-log10(pmax(metric_reg_res$padj, .Machine$double.xmin)), na.rm = TRUE) + 0.5

  volcano_plot <- EnhancedVolcano(metric_reg_res,
                                  lab = metric_reg_res$gene,
                                  selectLab = select_lab,
                                  drawConnectors = TRUE,
                                  maxoverlapsConnectors = Inf,
                                  labSize = 3.5,
                                  x = 'beta',
                                  y = 'padj',
                                  pointSize = 1,
                                  subtitle = NULL,
                                  FCcutoff = 0,
                                  xlim = quantile(metric_reg_res$beta, c(0.001, 0.999)),
                                  ylim = c(0, max_y),
                                  pCutoff = 0.05,
                                  caption = NULL,
                                  xlab = 'Regression coefficient',
                                  ylab = bquote(~-Log[10] ~ 'adj. p value'),
                                  axisLabSize = 15,
                                  colCustom = points_cols,
                                  arrowheads = FALSE,
                                  colAlpha = 1,
                                  legendPosition = 'none',
                                  title = paste(curr_metric, 'regression plot')) +
    theme(plot.title = element_text(hjust = 0.5))

  ggsave(file.path(OUTPUT_DIR, paste0(SAMPLE_NAME, "_volcano_", curr_metric, ".png")),
         volcano_plot, width = 8, height = 6, dpi = 150)
  ggsave(file.path(OUTPUT_DIR, paste0(SAMPLE_NAME, "_volcano_", curr_metric, ".pdf")),
         volcano_plot, width = 8, height = 6)
  print(volcano_plot)
}

#### Plot spatially regressed residuals for a single gene and mechanical property

In [ ]:
metric_use <- 'pressure'
gene <- dplyr::arrange(gsem_reg_res_smooth[gsem_reg_res_smooth$metric == metric_use, ], padj)[1, 'gene']

gsem_reg_res_singlegene <- do_gsem_regression(gene, metric_use, tensionmap_res, gex_res_dense,
  k_sp = K_SPATIAL, model_fx = MODEL_FX, use_bam = USE_BAM, method = gSEM_METHOD)

plot_df <- data.frame(
  r_X    = gsem_reg_res_singlegene[[2]],
  r_Y    = gsem_reg_res_singlegene[[3]],
  fitted = gsem_reg_res_singlegene[[1]]$fitted.values
)
p <- ggplot(plot_df) +
  geom_point(aes(x = r_X, y = r_Y)) +
  geom_line(aes(x = r_X, y = fitted), color = 'red') +
  ggtitle(gene) +
  theme_classic() +
  xlab(bquote(r[q])) +
  ylab(bquote(r[g]))
p <- ggExtra::ggMarginal(p, type = "histogram")
p

### v2 EXTENDED: what does cell shape find that stress does not?

The point of adding morphology is not to re-find the stress hits. This section isolates the
genes that only the shape metrics call significant, and reports the residual correlation
structure that determines whether those calls are independent evidence or the same signal
arriving through a correlated channel.

Outputs written to `OUTPUT_DIR`:

- `*_v2ext_shape_only_genes.csv` — genes significant on ≥1 shape metric and no mechanical metric
- `*_v2ext_residual_collinearity.png/.csv` — correlation between spatial residuals `r_metric`

In [ ]:
if (INCLUDE_SHAPE_METRICS && length(setdiff(metrics_use, metrics_mech)) > 0) {

  shape_metrics_run <- setdiff(metrics_use, metrics_mech)

  sig <- gsem_reg_res_smooth[!is.na(gsem_reg_res_smooth$padj) &
                              gsem_reg_res_smooth$padj < 0.05, ]

  genes_mech  <- unique(sig$gene[sig$metric %in% metrics_mech])
  genes_shape <- unique(sig$gene[sig$metric %in% shape_metrics_run])
  shape_only  <- setdiff(genes_shape, genes_mech)
  mech_only   <- setdiff(genes_mech,  genes_shape)
  both        <- intersect(genes_mech, genes_shape)

  cat(sprintf("Significant genes (padj < 0.05)\n"))
  cat(sprintf("  mechanical only : %d\n", length(mech_only)))
  cat(sprintf("  shape only      : %d\n", length(shape_only)))
  cat(sprintf("  both            : %d\n", length(both)))
  cat(sprintf("  union           : %d of %d tested (%.1f%%)\n",
              length(union(genes_mech, genes_shape)), length(genes_use),
              100 * length(union(genes_mech, genes_shape)) / length(genes_use)))

  if (length(shape_only) > 0) {
    so <- sig[sig$gene %in% shape_only & sig$metric %in% shape_metrics_run, ]
    so <- so[order(so$padj), c("gene", "metric", "beta", "pval", "padj")]
    write.csv(so, file.path(OUTPUT_DIR,
              paste0(SAMPLE_NAME, "_v2ext_shape_only_genes.csv")), row.names = FALSE)
    cat("\nTop shape-only associations:\n")
    print(utils::head(so, 20))
  } else {
    cat("\nNo genes were called by a shape metric alone.\n")
  }

  # ── Residual collinearity: the correlation that actually matters ───────────
  r_path <- file.path(RESIDUALS_DIR, "r_metrics.rds")
  if (!is.null(RESIDUALS_DIR) && file.exists(r_path)) {
    r_m  <- readRDS(r_path)
    rmat <- cor(do.call(cbind, r_m), method = "spearman", use = "complete.obs")

    rl <- expand.grid(a = rownames(rmat), b = colnames(rmat), stringsAsFactors = FALSE)
    rl$rho <- as.vector(rmat)

    p_res <- ggplot(rl, aes(x = a, y = b, fill = rho)) +
      geom_tile(colour = "white") +
      geom_text(aes(label = sprintf("%.2f", rho)), size = 2.8) +
      scale_fill_gradient2(low = "#2166AC", mid = "white", high = "#B2182B",
                           midpoint = 0, limits = c(-1, 1), name = "Spearman") +
      labs(x = NULL, y = NULL,
           title = "Spatial-residual correlation between metrics") +
      theme_bw() +
      theme(axis.text.x = element_text(angle = 45, hjust = 1),
            plot.title  = element_text(hjust = 0.5))

    ggsave(file.path(OUTPUT_DIR, paste0(SAMPLE_NAME, "_v2ext_residual_collinearity.png")),
           p_res, width = 8, height = 7, dpi = 150)
    write.csv(rmat, file.path(OUTPUT_DIR,
              paste0(SAMPLE_NAME, "_v2ext_residual_collinearity.csv")))
    print(p_res)

    cat("\nMax |rho| between any shape and any mechanical residual: ")
    cross <- abs(rmat[shape_metrics_run, metrics_mech, drop = FALSE])
    cat(sprintf("%.3f\n", max(cross, na.rm = TRUE)))
    cat("  (high values mean shape hits are largely re-reporting the stress signal)\n")
  } else {
    cat("\nRESIDUALS_DIR not populated - skipping residual collinearity.\n")
  }
}


#### `shape_index` and the rigidity transition

`shape_index` is the one morphology metric with an a-priori interpretable threshold: in the
2D vertex model, tissue with a mean shape index above ≈ 3.81 is unjammed/fluid-like, below it
jammed/solid-like. This cell reports where the sample sits and lists the strongest
shape-index-associated genes in each direction — positive β meaning expression rises with a
more fluid-like cell shape after spatial and area confounders are removed.

In [ ]:
if (INCLUDE_SHAPE_METRICS && "shape_index" %in% metrics_use) {

  si <- tensionmap_res$shape_index
  cat(sprintf("shape_index: mean=%.3f  median=%.3f  (rigidity transition = %.2f)\n",
              mean(si, na.rm = TRUE), median(si, na.rm = TRUE), SHAPE_INDEX_CRITICAL))
  cat(sprintf("  fluid-like (> %.2f): %.1f%% of cells\n",
              SHAPE_INDEX_CRITICAL, 100 * mean(si > SHAPE_INDEX_CRITICAL, na.rm = TRUE)))
  cat(sprintf("  tissue-level classification: %s\n\n",
              if (median(si, na.rm = TRUE) > SHAPE_INDEX_CRITICAL)
                "predominantly UNJAMMED / fluid-like" else
                "predominantly JAMMED / solid-like"))

  d <- gsem_reg_res_smooth[gsem_reg_res_smooth$metric == "shape_index" &
                            !is.na(gsem_reg_res_smooth$padj), ]
  d <- d[order(d$padj), ]
  sig <- d[d$padj < 0.05, ]

  cat(sprintf("shape_index associations: %d significant of %d tested\n",
              nrow(sig), nrow(d)))
  if (nrow(sig) > 0) {
    cat("\nTop UP with fluid-like shape (beta > 0):\n")
    print(utils::head(sig[sig$beta > 0, c("gene", "beta", "padj")], 15))
    cat("\nTop UP with solid-like shape (beta < 0):\n")
    print(utils::head(sig[sig$beta < 0, c("gene", "beta", "padj")], 15))
  }
}


#### GO overrepresentation analysis

Generalised over `metrics_use`, so shape metrics get the same treatment as the mechanical
ones. The V5 version hard-coded four `enrichGO` calls; this loops, which is what lets the
metric set stay configurable.

In [ ]:
go_results <- list()

for (m in metrics_use) {
  hits <- unique(gsem_reg_res_smooth$gene[
    !is.na(gsem_reg_res_smooth$padj) &
    gsem_reg_res_smooth$padj < GO_PADJ &
    gsem_reg_res_smooth$metric == m])

  if (length(hits) < 5) {
    cat(sprintf("%-20s only %d gene(s) below padj < %.2f - skipping GO\n",
                m, length(hits), GO_PADJ))
    go_results[[m]] <- NULL
    next
  }

  go_results[[m]] <- tryCatch(
    enrichGO(gene = hits, universe = genes_use, OrgDb = ORG_DB, ont = "BP",
             keyType = "SYMBOL", minGSSize = 10, maxGSSize = 500, pvalueCutoff = 1),
    error = function(e) { message(sprintf("enrichGO failed for %s: %s",
                                          m, conditionMessage(e))); NULL })

  n_terms <- if (is.null(go_results[[m]])) 0 else nrow(go_results[[m]]@result)
  cat(sprintf("%-20s %4d input genes -> %d GO terms\n", m, length(hits), n_terms))
}


In [ ]:
for (m in names(go_results)) {
  res <- go_results[[m]]
  if (is.null(res) || nrow(res@result) == 0) {
    message(sprintf("No GO result for %s", m))
    next
  }
  print(makebarplot(res@result,
                    title = sprintf("%s GO enrichment", gsub("_", " ", m)), 10))
}


### Post-hoc cell-type analysis

Uses the per-cell spatial residuals saved during gSEM (`gsem_residuals/`) to estimate mechano-gene associations within each cell type and while blocking for cell type — without re-running the expensive spatial smooths.

Requires:
- `gsem_residuals/` directory produced by the gSEM cell above (`RESIDUALS_DIR`)
- `cell_annotations.csv` produced by MB_04 (must contain `cell_type` column indexed by Xenium barcode)

In [ ]:
# ── Load saved residuals ──────────────────────────────────────────────────────
r_metrics_saved <- readRDS(file.path(RESIDUALS_DIR, "r_metrics.rds"))
cell_order      <- readLines(file.path(RESIDUALS_DIR, "cell_order.txt"))

batch_files <- sort(list.files(RESIDUALS_DIR, pattern = "r_gene_batch_.*.rds",
                                full.names = TRUE))
r_gene <- do.call(cbind, lapply(batch_files, readRDS))  # n_cells x n_genes
rownames(r_gene) <- cell_order
cat(sprintf("r_gene loaded: %d cells x %d genes\n", nrow(r_gene), ncol(r_gene)))

# ── Join MB_04 cell annotations ───────────────────────────────────────────────
cell_annot_path <- file.path(ANALYSIS_DIR, "analysis", "fullslide",
                              "cell_annotation", "cell_annotations.csv")
# read.csv used instead of fread: fread uses external shell tools (wc/grep) that
# are absent from the conda R kernel PATH on macOS, causing exit code 127.
cell_annot <- read.csv(cell_annot_path, check.names = FALSE)

# Diagnostic: confirm ID formats align before joining
cat("cell_order[1:3]:\n"); print(head(cell_order, 3))
cat("cell_annot[[1]][1:3]:\n"); print(head(cell_annot[[1]], 3))

# Direct match: cell_order and cell_annot[[1]] are both full cell IDs
# (e.g. "R0C0_Y0_X0_dblkpbhn-1") — no prefix stripping needed
cell_types <- cell_annot$cell_type_auto[match(cell_order, cell_annot[[1]])]

cat(sprintf("Cell type join: %d / %d cells annotated\n",
            sum(!is.na(cell_types)), length(cell_types)))
print(table(cell_types, useNA = "always"))

In [ ]:
# ── Stratified lm(): separate beta per cell type ─────────────────────────────
# For each cell type T: lm(r_gene[T, gene] ~ r_metric[T, m])
# ~15 types × 503 genes × 4 metrics ≈ 30K lm() calls; typically < 5 min serial.

unique_types <- sort(unique(cell_types[!is.na(cell_types)]))

strat_results <- do.call(rbind, lapply(unique_types, function(ct) {
  idx <- which(cell_types == ct)
  do.call(rbind, lapply(colnames(r_gene), function(gene) {
    r_Y <- r_gene[idx, gene]
    do.call(rbind, lapply(names(r_metrics_saved), function(m) {
      r_X <- r_metrics_saved[[m]][idx]
      mod <- lm(r_Y ~ r_X)
      s   <- summary(mod)$coefficients
      data.frame(cell_type = ct, gene = gene, metric = m,
                 n_cells = length(idx),
                 beta    = coef(mod)[2],
                 pval    = if (nrow(s) >= 2) s[2, 4] else NA_real_,
                 stringsAsFactors = FALSE)
    }))
  }))
}))

# FDR within each cell_type × metric stratum
strat_results$padj <- NA_real_
for (ct in unique_types) {
  for (m in names(r_metrics_saved)) {
    idx <- strat_results$cell_type == ct & strat_results$metric == m
    strat_results$padj[idx] <- p.adjust(strat_results$pval[idx], method = PADJ_METHOD)
  }
}

out_strat <- file.path(OUTPUT_DIR, paste0(SAMPLE_NAME, "_gsem_stratified.csv"))
data.table::fwrite(strat_results, out_strat)
cat(sprintf("Stratified results: %d rows → %s\n", nrow(strat_results), out_strat))
head(strat_results)

In [ ]:
# ── Blocked lm(): cell_type as covariate on all cells ────────────────────────
# lm(r_gene ~ r_metric + cell_type): controls for cell-type main effect while
# estimating the slide-wide mechano-gene beta after cell-type adjustment.
# ~503 genes × 4 metrics = 2012 lm() calls on 428K cells; ~10 min serial.
# Parallelise with makeForkCluster if needed.

annotated  <- !is.na(cell_types)
ct_factor  <- factor(cell_types[annotated])

blocked_results <- do.call(rbind, lapply(colnames(r_gene), function(gene) {
  r_Y <- r_gene[annotated, gene]
  do.call(rbind, lapply(names(r_metrics_saved), function(m) {
    r_X <- r_metrics_saved[[m]][annotated]
    mod <- lm(r_Y ~ r_X + ct_factor)
    s   <- summary(mod)$coefficients
    data.frame(gene = gene, metric = m,
               beta = if ("r_X" %in% rownames(s)) s["r_X", 1] else NA_real_,
               pval = if ("r_X" %in% rownames(s)) s["r_X", 4] else NA_real_,
               stringsAsFactors = FALSE)
  }))
}))

# FDR per metric stratum
blocked_results$padj <- NA_real_
for (m in names(r_metrics_saved)) {
  idx <- blocked_results$metric == m
  blocked_results$padj[idx] <- p.adjust(blocked_results$pval[idx], method = PADJ_METHOD)
}

out_blocked <- file.path(OUTPUT_DIR, paste0(SAMPLE_NAME, "_gsem_blocked.csv"))
data.table::fwrite(blocked_results, out_blocked)
cat(sprintf("Blocked results: %d rows → %s\n", nrow(blocked_results), out_blocked))
head(blocked_results)

#### Post-hoc visualisations

In [ ]:
#### Plot 1 — Slide-wide vs cell-type-blocked beta correlation
# Tests whether cell-type blocking materially changes the mechano-gene betas.
# Points on the diagonal with rho ≈ 1 indicate cell type is not a major confounder.
comp_blocked <- merge(
  gsem_reg_res_smooth[, c('gene','metric','beta','padj')],
  blocked_results[, c('gene','metric','beta','padj')],
  by = c('gene','metric'), suffixes = c('_slidewide', '_blocked')
)

rho_labels <- do.call(rbind, lapply(unique(comp_blocked$metric), function(m) {
  d <- comp_blocked[comp_blocked$metric == m, ]
  r <- cor(d$beta_slidewide, d$beta_blocked, method='spearman', use='complete.obs')
  data.frame(metric=m, label=sprintf("rho=%.2f", r),
             x=quantile(d$beta_slidewide, 0.05, na.rm=TRUE),
             y=max(d$beta_blocked, na.rm=TRUE))
}))

p_blocked_cor <- ggplot(comp_blocked,
       aes(x=beta_slidewide, y=beta_blocked,
           color=-log10(pmax(padj_blocked, .Machine$double.xmin)))) +
  geom_point(size=0.6, alpha=0.6) +
  geom_abline(slope=1, intercept=0, linetype='dashed', color='grey50') +
  geom_smooth(method='lm', se=FALSE, color='red', linewidth=0.5) +
  geom_text(data=rho_labels, aes(x=x, y=y, label=label),
            inherit.aes=FALSE, hjust=0, size=3.5) +
  facet_wrap(~metric, scales='free') +
  scale_color_viridis_c(name='-log10(padj)\nblocked') +
  labs(x='beta (slide-wide)', y='beta (cell-type blocked)',
       title='Effect of cell-type blocking on mechano-gene betas') +
  theme_bw() + theme(plot.title=element_text(hjust=0.5))

ggsave(file.path(OUTPUT_DIR, paste0(SAMPLE_NAME, '_slidewide_vs_blocked.png')),
       p_blocked_cor, width=10, height=8, dpi=150)
p_blocked_cor

In [ ]:
#### Plot 2 — Slide-wide vs stratified beta, faceted by cell type
# Shows which cell types recapitulate the global association (rho ≈ 1, on diagonal)
# vs those that diverge. Change m_show to explore other metrics.
comp_strat <- merge(
  gsem_reg_res_smooth[, c('gene','metric','beta','padj')],
  strat_results[, c('cell_type','gene','metric','beta','padj','n_cells')],
  by=c('gene','metric'), suffixes=c('_slidewide','_strat')
)

rho_strat <- do.call(rbind, lapply(unique(comp_strat$cell_type), function(ct) {
  do.call(rbind, lapply(unique(comp_strat$metric), function(m) {
    d <- comp_strat[comp_strat$cell_type==ct & comp_strat$metric==m, ]
    r <- cor(d$beta_slidewide, d$beta_strat, method='spearman', use='complete.obs')
    data.frame(cell_type=ct, metric=m, rho=round(r,2),
               n_cells=d$n_cells[1],
               label=sprintf("rho=%.2f\nn=%d", r, d$n_cells[1]))
  }))
}))

m_show <- 'pressure'  # change to explore other metrics
d_show    <- comp_strat[comp_strat$metric == m_show, ]
rho_show  <- rho_strat[rho_strat$metric  == m_show, ]
rho_show$x <- min(d_show$beta_slidewide, na.rm=TRUE)
rho_show$y <- max(d_show$beta_strat,     na.rm=TRUE)

p_strat_cor <- ggplot(d_show, aes(x=beta_slidewide, y=beta_strat)) +
  geom_point(size=0.5, alpha=0.5, color='steelblue') +
  geom_abline(slope=1, intercept=0, linetype='dashed', color='grey50') +
  geom_smooth(method='lm', se=FALSE, color='red', linewidth=0.5) +
  geom_text(data=rho_show, aes(x=x, y=y, label=label),
            inherit.aes=FALSE, hjust=0, vjust=1, size=2.8) +
  facet_wrap(~cell_type, scales='free') +
  labs(x='beta (slide-wide)', y='beta (stratified)',
       title=sprintf('Slide-wide vs per-cell-type betas: %s', m_show)) +
  theme_bw(base_size=9) + theme(plot.title=element_text(hjust=0.5))

ggsave(file.path(OUTPUT_DIR, paste0(SAMPLE_NAME, '_strat_vs_slidewide_', m_show, '.png')),
       p_strat_cor, width=max(8, length(unique_types)*1.5), height=max(6, ceiling(length(unique_types)/3)*2.5), dpi=150)
p_strat_cor

In [ ]:
#### Plot 4 — Significant mechano-gene hits per cell type
# Quick summary of which cell types have the most significant mechano-transcriptomic
# associations per metric. Bar height = n genes with padj < 0.05.
sig_counts <- do.call(rbind, lapply(unique_types, function(ct) {
  do.call(rbind, lapply(unique(strat_results$metric), function(m) {
    d <- strat_results[strat_results$cell_type==ct & strat_results$metric==m, ]
    data.frame(cell_type=ct, metric=m,
               n_sig   = sum(!is.na(d$padj) & d$padj < 0.05),
               n_cells = if (nrow(d) > 0) d$n_cells[1] else NA_integer_)
  }))
}))

# Order cell types by total significant hits across all metrics
ct_order <- names(sort(tapply(sig_counts$n_sig, sig_counts$cell_type, sum), decreasing=TRUE))
sig_counts$cell_type <- factor(sig_counts$cell_type, levels=ct_order)

p_sig <- ggplot(sig_counts, aes(x=cell_type, y=n_sig, fill=metric)) +
  geom_col(position='dodge') +
  scale_fill_manual(values=jdb_palette('corona')[seq_along(unique(sig_counts$metric))],
                    name='Metric') +
  labs(x=NULL, y='Significant genes (padj < 0.05)',
       title='Mechano-transcriptomic associations per cell type') +
  theme_bw() +
  theme(axis.text.x=element_text(angle=45, hjust=1),
        plot.title=element_text(hjust=0.5))

ggsave(file.path(OUTPUT_DIR, paste0(SAMPLE_NAME, '_sig_per_celltype.png')),
       p_sig, width=max(8, length(unique_types)*0.8), height=5, dpi=150)
p_sig

#### Blocked volcano plots


In [ ]:
#### Blocked volcano plots — cell-type-blocked gSEM (one plot per metric)
# blocked_results has the same shape as gsem_reg_res_smooth so EnhancedVolcano
# is used directly with identical formatting.

for (curr_metric in unique(blocked_results$metric)) {
  d <- dplyr::filter(blocked_results, metric == curr_metric)

  points_cols <- ifelse(d$padj < 0.05, 'red', 'lightgrey')
  names(points_cols) <- d$gene

  max_y <- max(-log10(pmax(d$padj, .Machine$double.xmin)), na.rm=TRUE) + 0.5

  vp <- EnhancedVolcano(d,
          lab             = d$gene,
          drawConnectors  = TRUE,
          x               = 'beta',
          y               = 'padj',
          pointSize       = 1,
          subtitle        = NULL,
          FCcutoff        = 0,
          xlim            = quantile(d$beta, c(0.001, 0.999)),
          ylim            = c(0, max_y),
          pCutoff         = 0.05,
          caption         = NULL,
          xlab            = 'Regression coefficient',
          ylab            = bquote(~-Log[10] ~ 'adj. p value'),
          axisLabSize     = 15,
          colCustom       = points_cols,
          arrowheads      = FALSE,
          colAlpha        = 1,
          legendPosition  = 'none',
          title           = paste(curr_metric, 'regression plot (cell-type blocked)')) +
    theme(plot.title = element_text(hjust = 0.5))

  ggsave(file.path(OUTPUT_DIR,
           paste0(SAMPLE_NAME, '_volcano_blocked_', curr_metric, '.png')),
         vp, width=8, height=6, dpi=150)
  print(vp)
}


#### Stratified volcano plots


In [ ]:
#### Stratified volcano plots — one ggplot per metric, faceted by cell type
# Same axis labels and colour scheme as the slide-wide EnhancedVolcano cells.
# Top N_LABEL genes per panel are labelled (ggrepel); grey dashed lines mark
# beta=0 and the padj=0.05 significance threshold.
suppressMessages(library(ggrepel))

N_LABEL <- 10   # max gene labels per cell-type panel

for (curr_metric in unique(strat_results$metric)) {
  d <- strat_results[strat_results$metric == curr_metric, ]
  d$neg_log10_padj <- -log10(pmax(d$padj, .Machine$double.xmin))
  d$Sig <- ifelse(!is.na(d$padj) & d$padj < 0.05, 'Significant', 'NS')

  # Label top N genes per cell type by padj
  d$label <- NA_character_
  for (ct in unique(d$cell_type)) {
    idx     <- which(d$cell_type == ct)
    top_idx <- idx[order(d$padj[idx], na.last=TRUE)[seq_len(min(N_LABEL, length(idx)))]]
    d$label[top_idx] <- d$gene[top_idx]
  }

  xlim_vals <- quantile(d$beta, c(0.001, 0.999), na.rm=TRUE)
  max_y     <- max(d$neg_log10_padj[is.finite(d$neg_log10_padj)], na.rm=TRUE) + 0.5

  p <- ggplot(d, aes(x=beta, y=neg_log10_padj, colour=Sig)) +
    geom_point(size=0.5, alpha=0.7) +
    geom_text_repel(aes(label=label), size=2, max.overlaps=15,
                    na.rm=TRUE, segment.size=0.2, show.legend=FALSE) +
    geom_vline(xintercept=0, linetype='dashed', colour='grey60', linewidth=0.3) +
    geom_hline(yintercept=-log10(0.05), linetype='dashed', colour='grey60', linewidth=0.3) +
    scale_colour_manual(values=c('Significant'='red', 'NS'='lightgrey'), name=NULL) +
    facet_wrap(~cell_type, scales='free') +
    coord_cartesian(xlim=xlim_vals, ylim=c(0, max_y)) +
    labs(x='Regression coefficient',
         y=bquote(~-Log[10] ~ 'adj. p value'),
         title=paste(curr_metric, '\u2014 stratified regression by cell type')) +
    theme_bw(base_size=9) +
    theme(plot.title=element_text(hjust=0.5),
          legend.position='none',
          strip.text=element_text(size=7))

  n_types <- length(unique(d$cell_type))
  ncols   <- ceiling(sqrt(n_types))
  nrows   <- ceiling(n_types / ncols)

  ggsave(file.path(OUTPUT_DIR,
           paste0(SAMPLE_NAME, '_volcano_stratified_', curr_metric, '.png')),
         p, width=ncols*3, height=nrows*3, dpi=150)
  print(p)
}


### Compare spatial regression to linear regression model

> **Note:** this section is disabled by default (`RUN_LINEAR_REGRESSION <- FALSE` in the config). Set it to `TRUE` to run the gSEM vs simple linear regression comparison.

#### Define functions for parallel linear regression

In [ ]:
if (RUN_LINEAR_REGRESSION) {
  lin_reg_res <- parallel_linear_regression(genes_use, tensionmap_res, gex_res_dense,
                                             ncores = NCORES)
}
rm(gex_res_dense); gc()  # recover ~6 GB after gSEM + LR (frees dense matrix regardless of LR flag)

In [ ]:
if (RUN_LINEAR_REGRESSION) lin_reg_res %>% head()

#### Compare coefficients for spatial vs linear regression

In [ ]:
if (RUN_LINEAR_REGRESSION) {
  comp_df <- merge(gsem_reg_res_smooth, lin_reg_res, by.x=c('gene','metric'), by.y=c('gene','metric'), suffixes=c('_gsem','_linreg'))
}

In [ ]:
if (RUN_LINEAR_REGRESSION) {
  all_cor <- cor.test(comp_df$beta_gsem, comp_df$beta_linreg, method = 'spearman')
  grob <- grobTree(textGrob(paste0("All genes:\n   rho = ",signif(all_cor$estimate,2),", p = ",signif(all_cor$p.value,2)), 
                              hjust=0, vjust=1, x=0.05,  y=0.95,
                              gp=gpar(col="black", fontface="italic", fontsize=12)))

  ggplot() +
      geom_point(data=comp_df, aes(x=beta_linreg, y=beta_gsem, color=metric)) +
      geom_vline(xintercept = 0) +
      geom_hline(yintercept = 0) + 
      theme_minimal() +
      annotation_custom(grob) +
      geom_abline(slope=1, intercept=0, linetype="dashed", color="grey") +
      ylab(bquote(beta[spatial])) +
      xlab(bquote(beta[linreg])) +
      scale_color_manual(values=jdb_palette('corona'),name = "Mechanical feature") +
      scale_alpha_continuous(name = bquote(~-Log[10] ~ 'p-value')) +
      theme(axis.title=element_text(size=16))
}

## Environment information:


In [ ]:
sessionInfo()